# Lead Scoring Machine Learning Project

Welcome to the **Lead Scoring Machine Learning Project**!
The goal of this project is to build a Logistic Regression model that predicts whether a lead will convert. We will analyze the **Leads X Education** dataset, engineer features, evaluate a model, and assign a **Lead Score (0-100)**.
Let's dive in! 🚀

## 1. Data Loading
- Load the dataset using pandas
- Show the first few rows
- Display shape, column names, and data types

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

# 1. Load the dataset
df = pd.read_csv('Leads X Education.csv')

# 2. Show the first few rows
display(df.head())

# 3. Display shape, columns, and data types
print('Dataset Shape:', df.shape)
print('\nColumn Names:\n', df.columns.tolist())
print('\nData Types:\n', df.dtypes)

## 2. Data Understanding
Before diving into the code, let's understand the key features of this dataset:

- **Converted**: This is our **Target Variable**. `1` means the lead was successfully converted (they bought the course), and `0` means they did not convert.
- **Lead Source**: The platform or method through which the lead found X Education (e.g., Google, Organic Search, Reference).
- **Total Time Spent on Website**: The total duration (in seconds) the lead spent browsing the website.
- **TotalVisits**: The total number of times the lead visited the website.
- **Last Activity**: The most recent action performed by the lead (e.g., Email Opened, SMS Sent).

In [ ]:
# Identify target variable and show distribution
target = 'Converted'
print('Distribution of Converted vs Non-Converted Leads:')
print(df[target].value_counts())
print('\nPercentage:\n', round(df[target].value_counts(normalize=True) * 100, 2))

plt.figure(figsize=(6, 4))
sns.countplot(x=target, data=df, palette='Set2')
plt.title('Distribution of Converted Leads (0 = No, 1 = Yes)')
plt.xlabel('Converted')
plt.ylabel('Count')
plt.show()

## 3. Exploratory Data Analysis (EDA)
Let's analyze lead behaviors and relationships using visualizations.
- Conversion rate by Lead Source and Last Activity.
- Average Website Time, Visits, and Page Views by Conversion status.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Conversion rate by Lead Source (Top 10)
top_sources = df['Lead Source'].value_counts().nlargest(10).index
sns.barplot(x='Lead Source', y='Converted', data=df[df['Lead Source'].isin(top_sources)], ax=axes[0,0], palette='viridis', ci=None)
axes[0,0].set_title('Conversion Rate by Lead Source (Top 10)')
axes[0,0].tick_params(axis='x', rotation=45)

# 2. Conversion rate by Last Activity (Top 10)
top_activities = df['Last Activity'].value_counts().nlargest(10).index
sns.barplot(x='Last Activity', y='Converted', data=df[df['Last Activity'].isin(top_activities)], ax=axes[0,1], palette='magma', ci=None)
axes[0,1].set_title('Conversion Rate by Last Activity (Top 10)')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Average Total Time Spent on Website by Converted
sns.boxplot(x='Converted', y='Total Time Spent on Website', data=df, ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Total Time Spent on Website vs Conversion')

# 4. Average Total Visits by Converted (Capping outliers for better viz)
visit_cap = df['TotalVisits'].quantile(0.95)
sns.boxplot(x='Converted', y='TotalVisits', data=df[df['TotalVisits'] < visit_cap], ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Total Visits vs Conversion (Filtered Outliers)')

plt.tight_layout()
plt.show()

# Display exact numerical averages
print('--- Average Behavioral Metrics ---')
print('Avg Time Spent on Website (Converted 1):', round(df[df['Converted']==1]['Total Time Spent on Website'].mean(), 2))
print('Avg Time Spent on Website (Converted 0):', round(df[df['Converted']==0]['Total Time Spent on Website'].mean(), 2))
print('\nAvg Total Visits (Converted 1):', round(df[df['Converted']==1]['TotalVisits'].mean(), 2))
print('Avg Total Visits (Converted 0):', round(df[df['Converted']==0]['TotalVisits'].mean(), 2))
print('\nAvg Page Views/Visit (Converted 1):', round(df[df['Converted']==1]['Page Views Per Visit'].mean(), 2))
print('Avg Page Views/Visit (Converted 0):', round(df[df['Converted']==0]['Page Views Per Visit'].mean(), 2))

## 4. Missing Value Analysis
- Calculate percentage of missing values.
- The dataset often uses `'Select'` to indicate that a customer hasn't selected an option. We will treat `'Select'` as `NaN`.
- Drop columns with very high missing values (>40%).
- Impute remaining missing values.

In [ ]:
# Treat 'Select' as null
df.replace('Select', np.nan, inplace=True)

# Calculate missing percentages
missing_pct = round(df.isnull().sum() / len(df) * 100, 2)
print('Missing Value Percentages (Top 15):\n', missing_pct[missing_pct > 0].sort_values(ascending=False).head(15))

# Drop columns with > 40% missing values
cols_to_drop = missing_pct[missing_pct > 40].index
df.drop(columns=cols_to_drop, inplace=True)
print(f'\nDropped {len(cols_to_drop)} columns with >40% missing values: {list(cols_to_drop)}')

# Impute missing values
# Numeric: Median. Categorical: Mode (most frequent)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

print('Remaining missing values in dataset:', df.isnull().sum().max())

## 5. Data Cleaning
- Remove identifier columns (`Prospect ID` and `Lead Number`) as they have no predictive power.
- Standardize binary text columns to numeric `1` and `0`.

In [ ]:
# Remove identifiers
df.drop(['Prospect ID', 'Lead Number'], axis=1, inplace=True, errors='ignore')

# Standardize binary categorical text columns directly
binary_cols = ['Do Not Email', 'Do Not Call', 'Search', 'Magazine', 'Newspaper Article',
               'X Education Forums', 'Newspaper', 'Digital Advertisement',
               'Through Recommendations', 'Receive More Updates About Our Courses',
               'Update me on Supply Chain Content', 'Get updates on DM Content',
               'I agree to pay the amount through cheque', 'A free copy of Mastering The Interview']

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: 1 if x == 'Yes' else 0)

print('Dataset shape after cleaning:', df.shape)

## 6. Feature Engineering
- Convert remaining categorical variables using One-Hot Encoding (`pd.get_dummies(drop_first=True)`).

In [ ]:
# Identify remaining categorical object columns
cat_cols = df.select_dtypes(include=['object']).columns

# One-Hot Enocding - drop_first=True avoids the dummy variable trap (multicollinearity)
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print('Dataset shape after One-Hot Encoding:', df.shape)
display(df.head(3))

## 7. Train-Test Split
Split the dataset into training (80%) and testing (20%) sets to ensure we evaluate the model on unseen data.

In [ ]:
# Separate Features (X) and Target (y)
X = df.drop('Converted', axis=1)
y = df['Converted']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Training set shape:', X_train.shape)
print('Testing set shape:', X_test.shape)

## 8. Model Building
- Scale the numeric features to ensure the Logistic Regression model converges quickly.- Initialize and fit a Logistic Regression model using `scikit-learn`.

In [ ]:
# Feature Scaling is important for linear models like Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train Logistic Regression
# max_iter=1000 ensures the algorithm converges successfully
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print('✅ Logistic Regression model trained successfully!')

## 9. Model Evaluation
Evaluate the model's performance on the 20% test set using key classification metrics.

### **Understanding the Confusion Matrix:**
- **True Positives (Predicted 1, Actual 1)**: Correctly predicted lead WILL convert.
- **True Negatives (Predicted 0, Actual 0)**: Correctly predicted lead WILL NOT convert.
- **False Positives (Predicted 1, Actual 0)**: Wrongly predicted lead will convert (Wasted sales effort).
- **False Negatives (Predicted 0, Actual 1)**: Wrongly predicted lead will NOT convert (Missed opportunity).

In [ ]:
# Predictions
y_pred = model.predict(X_test_scaled)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')

# Visualizing the Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.ylabel('Actual Classification')
plt.xlabel('Predicted Classification')
plt.show()

## 10. Lead Score Generation
Instead of just outputting `1` or `0`, we use `model.predict_proba()` to generate a probability of conversion. We scale this probability by 100 to generate a **Lead Score (0-100)**.

In [ ]:
# Get probabilities of conversion for the test set
# predict_proba returns [prob_0, prob_1]. We want prob_1.
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Create Lead Score (Scale 0-100)
lead_scores = np.round(y_pred_proba * 100, 1)

# Visualize score distribution
plt.figure(figsize=(8, 5))
sns.histplot(lead_scores, bins=20, kde=True, color='teal')
plt.title('Distribution of Generated Lead Scores')
plt.xlabel('Lead Score (0 - 100)')
plt.ylabel('Frequency of Leads')
plt.show()

# DataFrame showing subset of scores
results_df = pd.DataFrame({
    'Actual Conversion': y_test,
    'Predicted Prob.': np.round(y_pred_proba, 3),
    'Assigned Lead Score': lead_scores
})
display(results_df.head(10))

## 11. Feature Importance
Let's extract the Logistic Regression coefficients to see which features most strongly impact conversions.

In [ ]:
# Map coefficients back to feature names
feature_names = X.columns
coefficients = model.coef_[0]

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
})

# Get top 15 features by absolute importance
top_features = importance_df.sort_values(by='Abs_Coefficient', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=top_features, palette='coolwarm')
plt.title('Top 15 Most Important Features for Lead Conversion')
plt.show()

## 12. Business Insights & Actionable Conclusions

Based on our Lead Scoring model and EDA, here are the key takeaways for the sales and marketing teams:

### Which lead sources convert the most?
- High-Intent Referrals: **Reference** and **Welingak Institute** show massive positive correlations with conversion. These organic, highly trusted channels yield the best quality leads.
- Leads derived directly from **Lead Add Forms** signify explicit intent and heavily outperform ambiguous API or landing page submissions.

### Which behaviors indicate high conversion probability?
- **Total Time Spent on Website:** A massive indicator of intent. Leads that thoroughly read X Education's material instead of bouncing quickly are much more likely to commit.
- **Last Notable Activity - SMS Sent:** Engagement metrics highlight that sending a follow-up SMS strongly pushes leads toward conversion.
- **Working Professionals:** Leads specifying their occupation as 'Working Professional' convert much faster than unemployed individuals (likely due to budget and upskilling goals).

### How should Sales Teams prioritize leads?
With our newly developed **Lead Score (0-100)**, sales teams should adopt the following approach:
1. **Hot Leads (Score > 80):** High immediate priority. Reps should call these prospects within 5 minutes of form submission.
2. **Warm Leads (Score 40-79):** Significant potential via nurturing. Filter these to marketing for automated email/SMS drips. Reps should follow up weekly.
3. **Cold Leads (Score < 40):** Very low historic probability of converting (e.g. they opted for 'Do Not Email' or bounced quickly). Sales resources should not be directly wasted here until they are reactivated via organic means.

---

By adopting this systematic pipeline, X Education can reduce wasted sales calling efforts while capturing more motivated, high-converting prospects! 🙌